In [ ]:

import pyodbc
import ollama


# ==================================================
# 1. SQL SERVER CONNECTION
# ==================================================

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-NB27H6F;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# ==================================================
# 2. FUNCTION TO FETCH SQL DATA
# ==================================================

def get_data(query):

    cursor.execute(query)

    columns = [column[0] for column in cursor.description]

    rows = cursor.fetchall()

    result = []

    for row in rows:

        record = {}

        for i, column in enumerate(columns):
            record[column] = row[i]

        result.append(record)

    return result


# ==================================================
# 3. FORMAT SQL DATA
# ==================================================

def format_data(data):

    if not data:
        return "No records found."

    lines = []

    for record in data:

        row_data = []

        for key, value in record.items():

            row_data.append(f"{key}: {value}")

        lines.append(" | ".join(row_data))

    return "\n".join(lines)


# ==================================================
# 4. VENDOR-WISE EXPENSE SUMMARY
# ==================================================

def get_vendor_summary():

    query = """
    WITH PaymentTotals AS
    (
        SELECT
            expenseid,

            SUM(
                CASE
                    WHEN status = 'Paid'
                    THEN paidamount
                    ELSE 0
                END
            ) AS totalpaid

        FROM payment

        GROUP BY expenseid
    )

    SELECT
        COALESCE(v.vendorname, 'Unassigned / No vendor')
            AS vendorname,

        COUNT(e.expenseid) AS expensecount,

        ISNULL(SUM(e.amount), 0) AS totalexpense,

        ISNULL(SUM(p.totalpaid), 0) AS totalpaid,

        ISNULL(
            SUM(e.amount - ISNULL(p.totalpaid, 0)), 0
        ) AS outstandingamount,

        ISNULL(AVG(e.amount), 0) AS averageexpense,

        ISNULL(MAX(e.amount), 0) AS highestexpense,

        ISNULL(MIN(e.amount), 0) AS lowestexpense

    FROM vendor v

    FULL OUTER JOIN expense e
        ON v.vendorid = e.vendorid

    LEFT JOIN PaymentTotals p
        ON e.expenseid = p.expenseid

    GROUP BY v.vendorname

    ORDER BY totalexpense DESC;
    """

    return get_data(query)


# ==================================================
# 5. MONTHLY VENDOR-WISE EXPENSE ANALYSIS
# ==================================================

def get_monthly_vendor_analysis():

    query = """
    SELECT
        COALESCE(v.vendorname, 'Unassigned / No vendor')
            AS vendorname,

        YEAR(e.expensedate) AS expenseyear,

        MONTH(e.expensedate) AS expensemonth,

        COUNT(e.expenseid) AS expensecount,

        SUM(e.amount) AS totalexpense,

        AVG(e.amount) AS averageexpense

    FROM expense e

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    GROUP BY
        v.vendorname,
        YEAR(e.expensedate),
        MONTH(e.expensedate)

    ORDER BY
        expenseyear,
        expensemonth,
        totalexpense DESC;
    """

    return get_data(query)


# ==================================================
# 6. CATEGORY-WISE VENDOR ANALYSIS
# ==================================================

def get_vendor_category_analysis():

    query = """
    SELECT
        COALESCE(v.vendorname, 'Unassigned / No vendor')
            AS vendorname,

        ec.categoryname,

        COUNT(e.expenseid) AS expensecount,

        SUM(e.amount) AS totalexpense,

        AVG(e.amount) AS averageexpense

    FROM expense e

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    INNER JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    GROUP BY
        v.vendorname,
        ec.categoryname

    ORDER BY
        vendorname,
        totalexpense DESC;
    """

    return get_data(query)


# ==================================================
# 7. TOP 10 VENDOR EXPENSES
# ==================================================

def get_top_vendor_expenses():

    query = """
    SELECT TOP 10

        e.expenseid,

        COALESCE(v.vendorname, 'Unassigned / No vendor')
            AS vendorname,

        ec.categoryname,

        e.expensedate,

        e.description,

        e.amount

    FROM expense e

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    INNER JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    ORDER BY e.amount DESC;
    """

    return get_data(query)


# ==================================================
# 8. VENDOR PAYMENT STATUS ANALYSIS
# ==================================================

def get_vendor_payment_analysis():

    query = """
    WITH PaymentTotals AS
    (
        SELECT
            expenseid,

            SUM(
                CASE
                    WHEN status = 'Paid'
                    THEN paidamount
                    ELSE 0
                END
            ) AS totalpaid,

            SUM(
                CASE
                    WHEN status = 'Pending'
                    THEN paidamount
                    ELSE 0
                END
            ) AS pendingamount,

            SUM(
                CASE
                    WHEN status = 'Failed'
                    THEN paidamount
                    ELSE 0
                END
            ) AS failedamount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        COALESCE(v.vendorname, 'Unassigned / No vendor')
            AS vendorname,

        COUNT(e.expenseid) AS expensecount,

        ISNULL(SUM(e.amount), 0) AS totalexpense,

        ISNULL(SUM(p.totalpaid), 0) AS totalpaid,

        ISNULL(SUM(p.pendingamount), 0) AS pendingpayment,

        ISNULL(SUM(p.failedamount), 0) AS failedpayment,

        ISNULL(
            SUM(e.amount - ISNULL(p.totalpaid, 0)), 0
        ) AS outstandingamount

    FROM expense e

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    LEFT JOIN PaymentTotals p
        ON e.expenseid = p.expenseid

    GROUP BY v.vendorname

    ORDER BY totalexpense DESC;
    """

    return get_data(query)


# ==================================================
# 9. VENDOR INFORMATION
# ==================================================

def get_vendor_information():

    query = """
    SELECT
        vendorid,
        vendorname,
        gstin,
        contact,
        email,
        address

    FROM vendor

    ORDER BY vendorname;
    """

    return get_data(query)


# ==================================================
# 10. LOAD DATABASE INFORMATION
# ==================================================

def load_vendor_data():

    print("\nLoading vendor information from SQL Server...")

    vendor_summary = get_vendor_summary()

    monthly_analysis = get_monthly_vendor_analysis()

    category_analysis = get_vendor_category_analysis()

    top_expenses = get_top_vendor_expenses()

    payment_analysis = get_vendor_payment_analysis()

    vendor_information = get_vendor_information()

    all_data = f"""

    VENDOR INFORMATION
    ------------------
    {format_data(vendor_information)}


    VENDOR-WISE EXPENSE SUMMARY
    --------------------------
    {format_data(vendor_summary)}


    MONTHLY VENDOR ANALYSIS
    ----------------------
    {format_data(monthly_analysis)}


    CATEGORY-WISE VENDOR ANALYSIS
    -----------------------------
    {format_data(category_analysis)}


    TOP 10 VENDOR EXPENSES
    ---------------------
    {format_data(top_expenses)}


    VENDOR PAYMENT ANALYSIS
    -----------------------
    {format_data(payment_analysis)}

    """

    return all_data


# ==================================================
# 11. VENDOR ANALYSIS AGENT
# ==================================================

def vendor_analysis_agent(user_question):

    all_data = load_vendor_data()

    system_prompt = """
    You are an expert Vendor Analysis Agent.

    You analyse vendor expenses and payments using
    SQL Server business data.

    YOUR RESPONSIBILITIES:

    1. Vendor-wise total expenses.
    2. Vendor-wise monthly expenses.
    3. Highest and lowest spending vendors.
    4. Average expense per vendor.
    5. Vendor-wise outstanding payments.
    6. Vendor-wise payment status.
    7. Category-wise spending by vendor.
    8. Identification of unusually high expenses.
    9. Vendor spending trends.
    10. Vendor cost reduction recommendations.
    11. Vendor comparison.
    12. Vendor information and contact details.

    IMPORTANT RULES:

    - Use only the supplied database information.
    - Never invent vendor names or financial figures.
    - Do not add payment amounts to expense amounts.
    - Expense amount represents the recorded expense.
    - Only payments with status 'Paid' count as paid.
    - Pending and failed payments must be reported separately.
    - Outstanding = expense amount minus successful payments.
    - If payments exceed expenses, report the excess separately.
    - Do not assume a vendor is expensive without comparing
      comparable expense categories or periods.
    - Clearly mention when data is insufficient.
    - Show financial figures with two decimal places.
    - Give practical business recommendations.
    - Use simple professional business language.
    """

    user_prompt = f"""
    DATABASE INFORMATION:

    {all_data}

    USER QUESTION:

    {user_question}

    Analyse the database information and answer the question.
    """

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    return response["message"]["content"]


# ==================================================
# 12. INTERACTIVE USER INTERFACE
# ==================================================

print("\n======================================")
print("       VENDOR ANALYSIS AGENT")
print("======================================")

print("Database: orderssystem")
print("AI Model: llama3.2")

print("\nType 'exit' to close the agent.")

while True:

    question = input("\nAsk your vendor analysis question: ")

    if question.lower().strip() == "exit":
        break

    if not question.strip():
        print("Please enter a question.")
        continue

    try:

        answer = vendor_analysis_agent(question)

        print("\nAI VENDOR ANALYSIS")
        print("--------------------------------------")

        print(answer)

        print("--------------------------------------")

    except Exception as e:

        print("\nError:", e)


# ==================================================
# 13. CLOSE DATABASE CONNECTION
# ==================================================

cursor.close()

connection.close()

print("\nVendor Analysis Agent stopped.")



       VENDOR ANALYSIS AGENT
Database: orderssystem
AI Model: llama3.2

Type 'exit' to close the agent.



Ask your vendor analysis question:  Which vendor has the highest total expense?



Loading vendor information from SQL Server...

AI VENDOR ANALYSIS
--------------------------------------
To determine which vendor has the highest total expense, we need to analyze the data provided in the Vendor-wise Expense Summary table.

From the table, we can see that the vendor with the highest total expense is ABC Properties, with a total expense of 24000.00.

Here is a summary of the top 3 vendors with the highest total expense:

1. ABC Properties - 24000.00
2. Electricity Provider - 4000.00
3. Travel Services - 2500.00

Therefore, ABC Properties has the highest total expense among all the vendors. 

This can be used to inform business decisions such as negotiating better rates with the vendor or considering alternative vendors to reduce expenses.
--------------------------------------



Ask your vendor analysis question:  Show total expenses for each vendor.



Loading vendor information from SQL Server...


In [2]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'SQL Server Native Client RDA 11.0']
